> ### ⚠️**실행 안내 및 데이터 공지**
>
> 1. **데이터 보안 및 용량:** 원본(raw) 데이터는 보안 및 대용량 문제로 저장소에 포함되지 않았습니다.
> 2. **전처리 단계:** 본 노트북(01~02)은 raw 데이터를 기반으로 작성되어 실행이 제한됩니다.
> 3. **실행 가능 지점:** 분석 재현 및 모델 실행은 **`03_data_preparation.ipynb`부터 가능**합니다. (`data/processed/` 사용)

# 01. EDA & 문제 정의 — 서울 에어비앤비 RevPAR 분석

**분석 목적:** 호스트가 통제 가능한 변수 및 자치구 특성이 RevPAR에 미치는 영향을 가설별로 검증합니다.

**핵심 질문:**
- RevPAR를 결정하는 핵심 변수는 무엇인가?
- 슈퍼호스트·숙소 유형·콘텐츠 품질이 RevPAR에 얼마나 영향을 주는가?

**접근 방법:**
- Active+Operating 서브셋(14,399건) 기준으로 H1~H8 가설 순차 검증
- 각 가설을 시각화 + 통계 검정으로 채택/기각 판단

**데이터:** 32,061개 리스팅 (2024-10-01 ~ 2025-09-30, TTM 12개월)  
**검증 가설:** H1(슈퍼호스트) · H2(숙소유형) · H3(콘텐츠) · H4(최소숙박) · H5(추가요금) · H6(공급) · H7(POI) · H8(인구특성)

In [ ]:
# 비활성(Dormant/Ghost) 숙소 포함 시 RevPAR 왜곡 -> Active+Operating만 분석 대상으로 제한
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 시각화 설정
sns.set_style('whitegrid')
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
else:
    plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# 경로
CSV_PATH      = '../data/interim/final_seoul_airbnb_cleaned.csv'
FEATURES_PATH = '../data/processed/seoul_airbnb_features.csv'
POP_PATH      = '../data/external/monthly_population.csv'

# KPI 벤치마크
KPI_REVPAR_ALL_MEDIAN    = 8868
KPI_REVPAR_ACTIVE_MEDIAN = 47850
KPI_SUPERHOST_PREMIUM    = 0.831

# 데이터 로드
df = pd.read_csv(CSV_PATH)
mask_ao = (df['refined_status'] == 'Active') & (df['operation_status'] == 'Operating')
df_ao = df[mask_ao].copy()
print(f"✅ 전체 데이터: {len(df):,}건 | Active+Operating: {len(df_ao):,}건")

## 1. 데이터 개요

In [ ]:
# 시장 전체 vs 운영 서브셋의 RevPAR 격차를 먼저 파악해야 가설 검증 기준이 설정됨
print("=" * 55)
print("📋 데이터 개요")
print("=" * 55)
print(f"  분석 대상 (Active+Operating): {len(df_ao):,}건 ({len(df_ao)/len(df)*100:.1f}%)")
print(f"  비활성 (Dormant/Ghost): {df['refined_status'].isin(['Dormant','Ghost']).sum():,}건 ({df['refined_status'].isin(['Dormant','Ghost']).mean()*100:.1f}%)")
print(f"  자치구 수: {df['district'].nunique()}개")
print(f"  room_type 유형: {df['room_type'].nunique()}개")
print()
print("  TTM RevPAR 분포:")
print(f"    전체 중위값:        ₩{df['ttm_revpar'].median():,.0f}")
print(f"    Active+Oper 중위값: ₩{df_ao['ttm_revpar'].median():,.0f}")
print(f"    Active+Oper 평균:   ₩{df_ao['ttm_revpar'].mean():,.0f}")
print(f"    0원 비율 (전체):    {(df['ttm_revpar']==0).mean()*100:.1f}%")
print()
print("  주요 결측치:")
missing = df.isnull().sum()
for col, cnt in missing[missing > 0].sort_values(ascending=False).items():
    print(f"    {col}: {cnt:,}건 ({cnt/len(df)*100:.1f}%)")
print("=" * 55)

In [ ]:
# Dormant 포함 전체 vs Active+Operating 분포 비교 -> 서브셋 선택 근거 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['ttm_revpar'], bins=50, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(KPI_REVPAR_ALL_MEDIAN, color='red', linestyle='--',
                label=f'중위값 ₩{KPI_REVPAR_ALL_MEDIAN:,.0f}')
axes[0].set_title('전체 32,061건 TTM RevPAR 분포\n(Dormant 포함)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('TTM RevPAR (₩)'); axes[0].set_ylabel('빈도')
axes[0].set_xlim(0, 500000); axes[0].legend()

axes[1].hist(df_ao['ttm_revpar'], bins=50, color='coral', alpha=0.7, edgecolor='white')
axes[1].axvline(KPI_REVPAR_ACTIVE_MEDIAN, color='darkred', linestyle='--',
                label=f'중위값 ₩{KPI_REVPAR_ACTIVE_MEDIAN:,.0f}')
axes[1].set_title('Active+Operating 14,399건 TTM RevPAR 분포', fontsize=12, fontweight='bold')
axes[1].set_xlabel('TTM RevPAR (₩)'); axes[1].set_ylabel('빈도')
axes[1].set_xlim(0, 500000); axes[1].legend()

plt.tight_layout()
plt.show()

## 2. H1: 슈퍼호스트 프리미엄

**가설:** 슈퍼호스트 + 즉시예약 조합이 가장 높은 RevPAR를 보인다.

In [ ]:
# H1: 슈퍼호스트 프리미엄 시각화 -- 분포 형태(violin)와 절대 수치(bar) 동시 확인
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

superhost_map = {True: '슈퍼호스트', False: '일반 호스트'}
df_ao_plot = df_ao.copy()
df_ao_plot['호스트 유형'] = df_ao_plot['superhost'].map(superhost_map)

sns.violinplot(data=df_ao_plot, x='호스트 유형', y='ttm_revpar',
               palette=['#FF6B35', '#2196F3'], ax=axes[0], cut=0)
axes[0].axhline(KPI_REVPAR_ACTIVE_MEDIAN, color='gray', linestyle='--', label='전체 중위값')
axes[0].set_title('슈퍼호스트 여부별 RevPAR 분포', fontsize=13, fontweight='bold')
axes[0].set_ylabel('TTM RevPAR (₩)'); axes[0].set_ylim(0, 300000); axes[0].legend()

sh_stats = df_ao.groupby('superhost')['ttm_revpar'].agg(['median', 'mean', 'count'])
sh_stats.index = ['일반 호스트', '슈퍼호스트']
bars = axes[1].bar(sh_stats.index, sh_stats['median'],
                    color=['#2196F3', '#FF6B35'], width=0.5)
axes[1].set_title(f'슈퍼호스트 RevPAR 중위값 비교\n(프리미엄: +{KPI_SUPERHOST_PREMIUM*100:.1f}%)',
                   fontsize=13, fontweight='bold')
axes[1].set_ylabel('TTM RevPAR 중위값 (₩)')
for bar, row in zip(bars, sh_stats.itertuples()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'₩{row.median:,.0f}\n(n={row.count:,})',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# RevPAR 분포는 우측 꼬리(양의 왜도) -> 정규성 미충족 -> Mann-Whitney U 비모수 검정 사용
sh_revpar    = df_ao[df_ao['superhost']==True]['ttm_revpar']
non_sh_revpar= df_ao[df_ao['superhost']==False]['ttm_revpar']
u_stat, p_val = stats.mannwhitneyu(sh_revpar, non_sh_revpar, alternative='greater')
print(f"H1 Mann-Whitney U test (슈퍼호스트 > 일반호스트)")
print(f"  U통계량: {u_stat:,.0f},  p-value: {p_val:.4e}")
print(f"  → {'✅ 유의미 (p<0.05)' if p_val < 0.05 else '❌ 유의미하지 않음'}")
print(f"  슈퍼호스트 중위값:  ₩{sh_revpar.median():,.0f}")
print(f"  일반호스트 중위값:  ₩{non_sh_revpar.median():,.0f}")
print(f"  프리미엄:           +{(sh_revpar.median()/non_sh_revpar.median()-1)*100:.1f}%")

In [ ]:
# 슈퍼호스트 단독 효과가 아닌 즉시예약 조합 시 추가 프리미엄이 존재하는지 확인
cross_stats = df_ao.groupby(['superhost', 'instant_book'])['ttm_revpar'].agg(
    median='median', count='count'
).reset_index()
cross_stats['그룹'] = cross_stats.apply(
    lambda r: f"{'슈퍼호스트' if r['superhost'] else '일반'}+{'즉시예약' if r['instant_book'] else '수동승인'}", axis=1
)
cross_stats = cross_stats.sort_values('median', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#FF6B35', '#FF9B35', '#2196F3', '#64B5F6']
bars = ax.bar(cross_stats['그룹'], cross_stats['median'], color=colors, width=0.6)
ax.axhline(KPI_REVPAR_ACTIVE_MEDIAN, color='gray', linestyle='--', linewidth=1.5,
            label=f'전체 중위값 ₩{KPI_REVPAR_ACTIVE_MEDIAN:,.0f}')
ax.set_title('슈퍼호스트 × 즉시예약 조합별 RevPAR 중위값', fontsize=13, fontweight='bold')
ax.set_ylabel('TTM RevPAR 중위값 (₩)')
for bar, row in zip(bars, cross_stats.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'₩{row.median:,.0f}\n(n={row.count:,})', ha='center', va='bottom',
            fontsize=9, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 3. H2: 숙소 유형(room_type)별 RevPAR 구조

**가설:** 숙소 유형에 따라 RevPAR 구조(가격 vs 점유율 드라이버)가 다르다.

In [ ]:
# H2: room_type별 RevPAR와 점유율의 관계 -- '가격 드라이버 vs 점유율 드라이버' 구조 파악
rt_stats = df_ao.groupby('room_type')['ttm_revpar'].agg(
    median='median', mean='mean', count='count'
).sort_values('median', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rt_order = rt_stats.index.tolist()
sns.boxplot(data=df_ao, x='room_type', y='ttm_revpar',
            order=rt_order, palette='Set2', ax=axes[0], showfliers=False)
axes[0].axhline(KPI_REVPAR_ACTIVE_MEDIAN, color='red', linestyle='--')
axes[0].set_title('숙소 유형별 RevPAR 분포 (이상치 제외)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('TTM RevPAR (₩)'); axes[0].set_xlabel('')
axes[0].set_ylim(0, 200000)

occ_stats = df_ao.groupby('room_type')['ttm_occupancy'].median()
ax2 = axes[1].twinx()
bars = axes[1].bar(rt_stats.index, rt_stats['median'],
                    color=sns.color_palette('Set2', len(rt_stats)), width=0.5, alpha=0.8)
ax2.plot(occ_stats.reindex(rt_order).values, 'ko-', markersize=8, linewidth=2, label='중위 점유율')
axes[1].set_title('숙소 유형별 RevPAR vs 점유율', fontsize=12, fontweight='bold')
axes[1].set_ylabel('TTM RevPAR 중위값 (₩)'); ax2.set_ylabel('TTM 점유율 중위값')
for bar, val in zip(bars, rt_stats['median']):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 500, f'₩{val:,.0f}',
                 ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print("\nroom_type별 통계:")
print(rt_stats.to_string())

In [ ]:
# 3개 이상 그룹 비교 -> Kruskal-Wallis (비모수 ANOVA 대안, 정규성 불필요)
groups = [df_ao[df_ao['room_type']==rt]['ttm_revpar'] for rt in rt_order]
h_stat, p_val = stats.kruskal(*groups)
print(f"H2 Kruskal-Wallis test (room_type 간 RevPAR 차이)")
print(f"  H통계량: {h_stat:.2f},  p-value: {p_val:.4e}")
print(f"  → {'✅ 유의미 (p<0.05) — room_type 간 RevPAR 분포 차이 존재' if p_val < 0.05 else '❌ 유의미하지 않음'}")

## 4. H3: 콘텐츠 품질 (사진 수·평점)

**가설:** 사진 수와 평점이 RevPAR와 정상관 관계를 가지며, 21-35장 구간이 최적이다.

In [ ]:
# H3: 사진 수/평점과 RevPAR의 방향성 확인 -- 색상으로 점유율과의 관계도 함께 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc1 = axes[0].scatter(df_ao['photos_count'], df_ao['ttm_revpar'],
                       alpha=0.3, s=10, c=df_ao['ttm_occupancy'], cmap='YlOrRd')
axes[0].set_xlim(0, 100); axes[0].set_ylim(0, 400000)
axes[0].set_title('사진 수 vs TTM RevPAR', fontsize=12, fontweight='bold')
axes[0].set_xlabel('사진 수'); axes[0].set_ylabel('TTM RevPAR (₩)')
plt.colorbar(sc1, ax=axes[0], label='점유율')

sc2 = axes[1].scatter(df_ao['rating_overall'], df_ao['ttm_revpar'],
                       alpha=0.3, s=10, c=df_ao['photos_count'], cmap='Blues')
axes[1].set_ylim(0, 400000)
axes[1].set_title('전체 평점 vs TTM RevPAR', fontsize=12, fontweight='bold')
axes[1].set_xlabel('전체 평점'); axes[1].set_ylabel('TTM RevPAR (₩)')
plt.colorbar(sc2, ax=axes[1], label='사진 수')
plt.tight_layout()
plt.show()

r_photos, p_photos = stats.spearmanr(
    df_ao['photos_count'].dropna(),
    df_ao.loc[df_ao['photos_count'].notna(), 'ttm_revpar'])
r_rating, p_rating = stats.spearmanr(
    df_ao['rating_overall'].dropna(),
    df_ao.loc[df_ao['rating_overall'].notna(), 'ttm_revpar'])
print(f"사진 수  Spearman rho={r_photos:.3f} (p={p_photos:.4f})")
print(f"전체 평점 Spearman rho={r_rating:.3f} (p={p_rating:.4f})")

In [ ]:
# 최적 사진 수 구간 탐색 -- 선형 관계가 아닌 구간별 패턴 확인을 위해 pd.cut으로 구간화
df_ao_photos = df_ao.dropna(subset=['photos_count']).copy()
df_ao_photos['사진_구간'] = pd.cut(df_ao_photos['photos_count'],
                                     bins=[0, 10, 20, 35, 60, 500],
                                     labels=['0-10장', '11-20장', '21-35장', '36-60장', '60장+'])

photo_stats = df_ao_photos.groupby('사진_구간', observed=False)['ttm_revpar'].agg(
    median='median', count='count')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(photo_stats.index.astype(str), photo_stats['median'],
               color=['#E3F2FD', '#90CAF9', '#42A5F5', '#1565C0', '#0D47A1'], width=0.6)
ax.axhline(KPI_REVPAR_ACTIVE_MEDIAN, color='red', linestyle='--', linewidth=1.5,
            label=f'전체 중위값 ₩{KPI_REVPAR_ACTIVE_MEDIAN:,.0f}')
ax.set_title('사진 수 구간별 TTM RevPAR 중위값\n(최적 구간: 21-35장)', fontsize=13, fontweight='bold')
ax.set_ylabel('TTM RevPAR 중위값 (₩)'); ax.set_xlabel('사진 수 구간')
for bar, row in zip(bars, photo_stats.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'₩{row.median:,.0f}\n(n={row.count:,})', ha='center', va='bottom',
            fontsize=9, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 5. H4: 최소 숙박일 최적점

**가설:** min_nights 2-3박 구간이 RevPAR 최적점이다.

In [ ]:
# H4: min_nights 극단값(30박 초과) 제외 후 구간별 중위값 비교
df_ao_mn = df_ao[df_ao['min_nights'] <= 30].copy()
df_ao_mn['최소박_구간'] = pd.cut(df_ao_mn['min_nights'],
                                   bins=[0, 1, 2, 3, 7, 14, 30],
                                   labels=['1박', '2박', '3박', '4-7박', '8-14박', '15-30박'])

mn_stats = df_ao_mn.groupby('최소박_구간', observed=False)['ttm_revpar'].agg(
    median='median', count='count')

fig, ax = plt.subplots(figsize=(10, 5))
colors_mn = ['#FFF176', '#FFEE58', '#FFD600', '#FF8F00', '#E65100', '#B71C1C']
bars = ax.bar(mn_stats.index.astype(str), mn_stats['median'],
               color=colors_mn, width=0.6, edgecolor='white')
ax.axhline(KPI_REVPAR_ACTIVE_MEDIAN, color='red', linestyle='--', linewidth=1.5,
            label=f'전체 중위값 ₩{KPI_REVPAR_ACTIVE_MEDIAN:,.0f}')
ax.set_title('최소 숙박일 수별 TTM RevPAR 중위값 (30박 이하)', fontsize=13, fontweight='bold')
ax.set_ylabel('TTM RevPAR 중위값 (₩)'); ax.set_xlabel('최소 숙박일 수 구간')
for bar, row in zip(bars, mn_stats.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'₩{row.median:,.0f}\n(n={row.count:,})', ha='center', va='bottom', fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 6개 구간 동시 비교 -> Kruskal-Wallis (비모수 다중 그룹 검정)
mn_groups = [df_ao_mn[df_ao_mn['최소박_구간']==g]['ttm_revpar']
             for g in ['1박', '2박', '3박', '4-7박', '8-14박', '15-30박']]
mn_groups = [g for g in mn_groups if len(g) > 0]
h_stat, p_val = stats.kruskal(*mn_groups)
print(f"H4 Kruskal-Wallis test (min_nights 구간 간 RevPAR 차이)")
print(f"  H통계량: {h_stat:.2f},  p-value: {p_val:.4e}")
print(f"  → {'✅ 유의미 (p<0.05) — 최소 숙박일 구간별 RevPAR 분포 차이 존재' if p_val < 0.05 else '❌ 유의미하지 않음'}")

## 6. H5: 추가 게스트 요금 정책

**가설:** 추가 요금 정책 활성화가 RevPAR를 높이며, 특히 대형 숙소에서 효과가 크다.

In [ ]:
# H5: 추가요금 정책 효과 -- 전체 영향과 침실 수별 교차 효과를 분리해서 확인
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

efp_stats = df_ao.groupby('extra_guest_fee_policy')['ttm_revpar'].agg(
    median='median', count='count')
labels = {0: '추가요금 없음', 1: '추가요금 있음'}
efp_stats.index = [labels[i] for i in efp_stats.index]
bars = axes[0].bar(efp_stats.index, efp_stats['median'],
                    color=['#B0BEC5', '#FF7043'], width=0.5)
axes[0].set_title('추가 요금 정책별 RevPAR 중위값', fontsize=12, fontweight='bold')
axes[0].set_ylabel('TTM RevPAR 중위값 (₩)')
for bar, row in zip(bars, efp_stats.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'₩{row.median:,.0f}\n(n={row.count:,})', ha='center', va='bottom', fontweight='bold')

bgrp_efp = df_ao.groupby(['bedrooms_group', 'extra_guest_fee_policy'])['ttm_revpar'].median().unstack()
bgrp_efp.columns = ['추가요금 없음', '추가요금 있음']
bgrp_efp.dropna().plot(kind='bar', ax=axes[1], color=['#B0BEC5', '#FF7043'], width=0.7, edgecolor='white')
axes[1].set_title('침실 수별 × 추가요금 정책 RevPAR', fontsize=12, fontweight='bold')
axes[1].set_ylabel('TTM RevPAR 중위값 (₩)'); axes[1].set_xlabel('침실 수 그룹')
axes[1].tick_params(axis='x', rotation=45); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 추가요금 유무 2개 그룹 비교 -> Mann-Whitney U (비정규 분포)
efp_0 = df_ao[df_ao['extra_guest_fee_policy']==0]['ttm_revpar']
efp_1 = df_ao[df_ao['extra_guest_fee_policy']==1]['ttm_revpar']
u_stat, p_val = stats.mannwhitneyu(efp_1, efp_0, alternative='greater')
print(f"H5 Mann-Whitney U test (추가요금 있음 > 없음)")
print(f"  U통계량: {u_stat:,.0f},  p-value: {p_val:.4e}")
print(f"  → {'✅ 유의미 (p<0.05)' if p_val < 0.05 else '❌ 유의미하지 않음'}")
print(f"  추가요금 있음 중위값: ₩{efp_1.median():,.0f}  (n={len(efp_1):,})")
print(f"  추가요금 없음 중위값: ₩{efp_0.median():,.0f}  (n={len(efp_0):,})")

## 7. H6: 자치구 공급 압력

**가설:** 공급 과잉 자치구(마포구 등)는 RevPAR 압박을 받는다.

In [ ]:
# H6: 자치구 단위 공급 압력 분석 -- 리스팅 수가 많은 구가 RevPAR 압박을 받는지 확인
df_feat = pd.read_csv(FEATURES_PATH)
df_feat_ao = df_feat[(df_feat['refined_status']=='Active') & (df_feat['operation_status']=='Operating')].copy()

district_agg = df_feat.groupby('district').agg(
    total_listings=('ttm_revpar', 'count'),
    dormant_count=('refined_status', lambda x: x.isin(['Dormant','Ghost']).sum()),
    superhost_rate=('superhost', 'mean'),
).reset_index()
district_agg['ao_revpar_median'] = df_feat_ao.groupby('district')['ttm_revpar'].median().reindex(district_agg['district']).values
district_agg['dormant_ratio'] = district_agg['dormant_count'] / district_agg['total_listings']
district_agg['supply_share'] = district_agg['total_listings'] / district_agg['total_listings'].sum()

print("공급량 상위 5개 자치구:")
print(district_agg[['district','total_listings','ao_revpar_median','dormant_ratio']].sort_values('total_listings', ascending=False).head(5).to_string(index=False))

In [ ]:
# 공급량 vs RevPAR 산점도 (버블=공급 규모, 색=비활성비율) -- 자치구 레이블로 패턴 식별
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc = axes[0].scatter(district_agg['total_listings'], district_agg['ao_revpar_median'],
                      s=district_agg['total_listings']/50, alpha=0.7,
                      c=district_agg['dormant_ratio'], cmap='RdYlGn_r')
for _, row in district_agg.iterrows():
    if row['total_listings'] > 2500 or row['ao_revpar_median'] > 50000:
        axes[0].annotate(row['district'], (row['total_listings'], row['ao_revpar_median']),
                         fontsize=7, ha='center', va='bottom')
plt.colorbar(sc, ax=axes[0], label='비활성 비율')
axes[0].set_title('자치구별 공급량 vs RevPAR\n(원=리스팅수, 색=비활성비율)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('총 리스팅 수'); axes[0].set_ylabel('중위 RevPAR (Active+Oper, ₩)')

top_d = district_agg.sort_values('ao_revpar_median', ascending=True)
colors_r = ['#E53935' if v > 0.6 else '#FB8C00' if v > 0.5 else '#43A047'
            for v in top_d['dormant_ratio']]
axes[1].barh(top_d['district'], top_d['ao_revpar_median'], color=colors_r, edgecolor='white')
axes[1].axvline(KPI_REVPAR_ACTIVE_MEDIAN, color='black', linestyle='--',
                label=f'전체 중위값 ₩{KPI_REVPAR_ACTIVE_MEDIAN:,.0f}')
axes[1].set_title('자치구별 RevPAR 순위\n(색=비활성비율: 빨강60%+, 주황50-60%, 초록50%-)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('중위 RevPAR (₩)'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

r_supply, p_supply = stats.spearmanr(district_agg['total_listings'].dropna(),
                                      district_agg['ao_revpar_median'].dropna())
print(f"H6 공급량 vs RevPAR  Spearman rho={r_supply:.3f} (p={p_supply:.4f})")

## 8. H7: POI 근접성

**가설:** 주요 관광지(POI)에 가까울수록 RevPAR가 높다.

In [ ]:
# H7: POI 거리 100m 단위 구간화 -- 임계 거리 확인을 위해 단순 상관 대신 구간화
df_ao_poi = df_ao.dropna(subset=['nearest_poi_dist_km']).copy()

bins   = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.5, 2.0, np.inf]
labels = ['~100m','100-200m','200-300m','300-400m','400-500m',
          '500-600m','600-700m','700-800m','800-900m','900m-1km',
          '1-1.5km','1.5-2km','2km+']
df_ao_poi['POI_구간'] = pd.cut(df_ao_poi['nearest_poi_dist_km'],
                               bins=bins, labels=labels, include_lowest=True)

poi_stats = df_ao_poi.groupby('POI_구간', observed=False)['ttm_revpar'].agg(
    median='median', count='count').reset_index()

plt.figure(figsize=(16, 6))
colors_poi = plt.cm.Greens(np.linspace(0.8, 0.2, len(poi_stats)))
bars = plt.bar(poi_stats['POI_구간'], poi_stats['median'],
               color=colors_poi, width=0.7, edgecolor='white', alpha=0.8)
plt.axhline(KPI_REVPAR_ACTIVE_MEDIAN, color='#FF5A5F', linestyle='--', linewidth=2,
            label=f'전체 중위값 ₩{KPI_REVPAR_ACTIVE_MEDIAN:,.0f}')
plt.title('관광지(POI) 거리 100m 단위별 RevPAR 중위값', fontsize=14, fontweight='bold', pad=20)
plt.ylabel('TTM RevPAR 중위값 (₩)'); plt.xlabel('가장 가까운 관광지까지의 거리')
plt.xticks(rotation=45)
for i, bar in enumerate(bars):
    val, cnt = poi_stats.loc[i, 'median'], poi_stats.loc[i, 'count']
    if cnt > 0:
        plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                 f'₩{int(val/1000)}k\n(n={cnt})', ha='center', va='bottom', fontsize=8)
plt.legend(); plt.tight_layout(); plt.show()

r_poi, p_poi = stats.spearmanr(df_ao_poi['nearest_poi_dist_km'], df_ao_poi['ttm_revpar'])
print(f"H7 POI 거리 vs RevPAR  Spearman rho={r_poi:.3f} (p={p_poi:.4f})")

## 9. H8: 인구 구조와 RevPAR 관계

**가설:** 주야간 인구 비율(업무형)과 외국인 비율(관광형)이 높은 자치구가 높은 RevPAR를 보인다.

In [ ]:
# H8: 자치구 인구 특성(주야간/외국인 비율) 분석 -- scatter_corr은 r값 자동 표기 포함
DISTRICT_MAP = {
    'Gangnam-gu':'강남구', 'Gangdong-gu':'강동구', 'Gangbuk-gu':'강북구',
    'Gangseo-gu':'강서구', 'Gwanak-gu':'관악구', 'Gwangjin-gu':'광진구',
    'Guro-gu':'구로구', 'Geumcheon-gu':'금천구', 'Nowon-gu':'노원구',
    'Dobong-gu':'도봉구', 'Dongdaemun-gu':'동대문구', 'Dongjak-gu':'동작구',
    'Mapo-gu':'마포구', 'Seodaemun-gu':'서대문구', 'Seocho-gu':'서초구',
    'Seongdong-gu':'성동구', 'Seongbuk-gu':'성북구', 'Songpa-gu':'송파구',
    'Yangcheon-gu':'양천구', 'Yeongdeungpo-gu':'영등포구', 'Yongsan-gu':'용산구',
    'Eunpyeong-gu':'은평구', 'Jongno-gu':'종로구', 'Jung-gu':'중구',
    'Jungnang-gu':'중랑구',
}
TYPE_COLORS = {'관광형':'#e74c3c', '업무형':'#3498db', '혼합형':'#f39c12', '주거형':'#2ecc71'}

def scatter_corr(ax, x, y, labels, xlabel, ylabel, title, color='steelblue'):
    ax.scatter(x, y, s=80, color=color, alpha=0.75, edgecolors='white', linewidth=0.8)
    for i, label in enumerate(labels):
        ax.annotate(label, (x.iloc[i], y.iloc[i]), fontsize=7.5,
                    xytext=(4, 2), textcoords='offset points')
    z = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, np.poly1d(z)(x_line), 'r--', linewidth=1.5, alpha=0.8)
    r, pval = stats.pearsonr(x, y)
    sig = '**' if pval < 0.05 else ('*' if pval < 0.10 else '')
    ax.set_xlabel(xlabel, fontsize=9); ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(f'{title}\nr = {r:.3f}{sig}  (p = {pval:.3f})', fontsize=10, fontweight='bold')
    ax.tick_params(labelsize=8)
    return r, pval

In [ ]:
# TTM 12개월 평균으로 계절성 평탄화 -- 단일 월 데이터 사용 시 계절 편향 발생
pop_df = pd.read_csv(POP_PATH)
pop_df.columns = ['year_month', 'district_kor', 'total_pop', 'domestic_pop',
                  'foreign_pop', 'day_pop', 'night_pop']
pop_clean = pop_df[pop_df['district_kor'] != '서울시'].copy()
pop_clean['year_month'] = pd.to_datetime(pop_clean['year_month'], format='%Y-%m')

# TTM 12개월 평균
ttm_pop = pop_clean.groupby('district_kor').agg(
    ttm_pop       =('total_pop',   'mean'),
    ttm_day_pop   =('day_pop',     'mean'),
    ttm_night_pop =('night_pop',   'mean'),
    ttm_foreign_pop=('foreign_pop','mean'),
).reset_index()

# L90 최근 3개월
l90_start = pop_clean['year_month'].max() - pd.DateOffset(months=2)
pop_l90 = pop_clean[pop_clean['year_month'] >= l90_start].groupby('district_kor').agg(
    l90_pop=('total_pop','mean')).reset_index()

pop_combined = ttm_pop.merge(pop_l90, on='district_kor', how='inner')
pop_combined['pop_growth_ratio'] = pop_combined['l90_pop'] / pop_combined['ttm_pop']
pop_combined['day_night_ratio']  = pop_combined['ttm_day_pop'] / pop_combined['ttm_night_pop']
pop_combined['foreign_ratio']    = pop_combined['ttm_foreign_pop'] / pop_combined['ttm_pop']
pop_combined['foreign_ratio_pct']= pop_combined['foreign_ratio'] * 100

# Operating 기준 자치구 집계 (RevPAR는 평균 사용)
df_oper = df[df['operation_status']=='Operating'].copy()
d_agg = df_oper.groupby('district').agg(
    avg_occupancy=('ttm_occupancy','mean'),
    avg_adr      =('ttm_avg_rate', 'mean'),
    avg_revpar   =('ttm_revpar',   'mean'),
    listing_count=('ttm_revpar',   'count'),
).reset_index()
d_agg['district_kor'] = d_agg['district'].map(DISTRICT_MAP)

# 병합
analysis_df = d_agg.merge(
    pop_combined.rename(columns={'district_kor': 'district_kor_pop'}),
    left_on='district_kor', right_on='district_kor_pop', how='inner'
)
analysis_df['listing_density'] = analysis_df['listing_count'] / analysis_df['ttm_pop'] * 10000
print(f"분석 테이블: {len(analysis_df)}개 자치구")

In [ ]:
# 인구 규모/밀도/성장률 3종 vs RevPAR -- 어떤 인구 지표가 RevPAR와 더 연관되는지 비교
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
r4, p4 = scatter_corr(axes[0], analysis_df['ttm_pop']/10000, analysis_df['avg_revpar'],
    analysis_df['district_kor'], 'TTM 평균 생활인구 (만 명)', '평균 RevPAR (원)',
    '인구 규모 vs RevPAR', color='steelblue')
r5, p5 = scatter_corr(axes[1], analysis_df['listing_density'], analysis_df['avg_revpar'],
    analysis_df['district_kor'], '숙소 밀도 (만 명당 숙소 수)', '평균 RevPAR (원)',
    '공급 밀도 vs RevPAR', color='darkorange')
r6, p6 = scatter_corr(axes[2], analysis_df['pop_growth_ratio'], analysis_df['avg_revpar'],
    analysis_df['district_kor'], '인구 성장 비율 (l90/ttm)', '평균 RevPAR (원)',
    '인구 성장 vs RevPAR', color='seagreen')
plt.tight_layout(); plt.show()

In [ ]:
# 중앙값 기준 2x2 사분면 분류 -- 관광형/업무형/혼합형/주거형 전략 그룹 도출
med_dn = analysis_df['day_night_ratio'].median()
med_fr = analysis_df['foreign_ratio'].median()

def classify(row):
    hd = row['day_night_ratio'] >= med_dn
    hf = row['foreign_ratio']   >= med_fr
    if hd and hf:     return '관광형'
    if hd and not hf: return '업무형'
    if not hd and hf: return '혼합형'
    return '주거형'

analysis_df['district_type'] = analysis_df.apply(classify, axis=1)
type_summary = analysis_df.groupby('district_type').agg(
    count        =('district_kor','count'),
    avg_occupancy=('avg_occupancy','mean'),
    avg_adr      =('avg_adr',     'mean'),
    avg_revpar   =('avg_revpar',  'mean'),
).reset_index()

print(f"분류 기준: day_night_ratio 중앙값={med_dn:.3f}, foreign_ratio 중앙값={med_fr:.5f}")
for dt in ['관광형','업무형','혼합형','주거형']:
    dnames = analysis_df[analysis_df['district_type']==dt]['district_kor'].tolist()
    row_ts = type_summary[type_summary['district_type']==dt].iloc[0]
    print(f"  {dt} ({len(dnames)}개): RevPAR {row_ts['avg_revpar']:,.0f}원 | Occ {row_ts['avg_occupancy']:.3f}")
    print(f"     {dnames}")

In [ ]:
# 유형별 점유율/ADR/RevPAR 비교 -- RevPAR 차이가 가격에서 오는지 점유율에서 오는지 분해
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, (col, label) in zip(axes, [
        ('avg_occupancy', '평균 Occupancy'),
        ('avg_adr',       '평균 ADR (원)'),
        ('avg_revpar',    '평균 RevPAR (원)')]):
    sorted_ts = type_summary.sort_values(col, ascending=False)
    bars_c = ax.bar(sorted_ts['district_type'], sorted_ts[col],
                    color=[TYPE_COLORS[t] for t in sorted_ts['district_type']],
                    edgecolor='white', linewidth=1)
    ax.set_title(f'유형별 {label}', fontsize=10, fontweight='bold')
    ax.set_ylim(0, sorted_ts[col].max()*1.28)
    for bar, val in zip(bars_c, sorted_ts[col]):
        fmt = f'{val:.3f}' if 'ccupancy' in label else f'{val/10000:.0f}만'
        ax.text(bar.get_x()+bar.get_width()/2, val*1.02, fmt,
                ha='center', fontsize=10, fontweight='bold')

lp = [mpatches.Patch(color=c, label=t) for t,c in TYPE_COLORS.items()]
fig.legend(handles=lp, loc='upper right', bbox_to_anchor=(0.99, 1.08), ncol=4, fontsize=9)
plt.suptitle('자치구 유형별 에어비앤비 성과 비교 (TTM 기준)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 버블 사분면 -- 자치구별 입지 전략 포지셔닝 (버블 크기=RevPAR 수준)
fig, ax = plt.subplots(figsize=(10, 8))
for dtype, grp in analysis_df.groupby('district_type'):
    ax.scatter(grp['day_night_ratio'], grp['foreign_ratio_pct'],
               s=grp['avg_revpar']/1200+60, color=TYPE_COLORS[dtype],
               alpha=0.82, edgecolors='white', linewidth=0.8, label=dtype)
    for _, row in grp.iterrows():
        ax.annotate(row['district_kor'], (row['day_night_ratio'], row['foreign_ratio_pct']),
                    fontsize=8, xytext=(4, 3), textcoords='offset points')
ax.axvline(med_dn, color='grey', linestyle='--', linewidth=1, alpha=0.6)
ax.axhline(med_fr*100, color='grey', linestyle='--', linewidth=1, alpha=0.6)
ax.set_xlabel('주야간 인구 비율 (주간/야간)', fontsize=11)
ax.set_ylabel('외국인 비율 (%)', fontsize=11)
ax.set_title('자치구 상권 구조 사분면\n(버블 크기=평균 RevPAR)', fontsize=12, fontweight='bold')
ax.legend(title='자치구 유형', fontsize=10)
plt.tight_layout(); plt.show()

## 10. 종합 결론

**가설별 검증 결과 요약**

In [ ]:
# 8개 가설 검증 결과를 채택/조건부/기각으로 정리 -- 핵심 근거 수치 포함
summary = [
    ('H1', '슈퍼호스트 프리미엄',     '✅ 채택', '+83.1% RevPAR, Mann-Whitney p<0.001'),
    ('H2', '숙소 유형별 RevPAR 차이', '✅ 채택', 'entire_home >> private_room, Kruskal-Wallis p<0.001'),
    ('H3', '사진·평점 정상관',         '✅ 채택', '21-35장 최적, Spearman rho>0'),
    ('H4', 'min_nights 2-3박 최적',    '✅ 채택', '구간별 유의미한 차이, Kruskal-Wallis p<0.001'),
    ('H5', '추가요금 정책 효과',        '⚠️ 조건부 채택', '대형 숙소에서 효과 뚜렷, Mann-Whitney 확인'),
    ('H6', '공급 과잉 → RevPAR 압박',  '⚠️ 약한 지지', '마포구: 최다 공급 but 중위 RevPAR 양호'),
    ('H7', 'POI 근접성 효과',           '⚠️ 약한 지지', '거리 증가 시 RevPAR 감소 경향, Spearman 약한 음의 상관'),
    ('H8', '인구 구조 × RevPAR',        '✅ 채택', '관광형(외국인↑) > 업무형 > 혼합형 > 주거형'),
]

print(f"{'가설':<5} {'내용':<22} {'결론':<18} {'근거'}")
print('-' * 85)
for row in summary:
    print(f"{row[0]:<5} {row[1]:<22} {row[2]:<18} {row[3]}")

## 비즈니스 임플리케이션

| 우선순위 | 액션 | 예상 효과 |
|---------|------|---------|
| ★★★ | 슈퍼호스트 달성 + 즉시예약 활성화 | RevPAR +83% (프리미엄 직접 효과) |
| ★★★ | 숙소를 entire_home으로 제공 | private_room 대비 2-3배 RevPAR |
| ★★☆ | 사진 21-35장 최적화 | 사진 0-10장 대비 유의미한 RevPAR 향상 |
| ★★☆ | 최소 숙박일 2-3박 설정 | 1박·7박+ 대비 RevPAR 최적화 |
| ★★☆ | 대형 숙소 추가요금 정책 활성화 | 2+ 침실에서 효과 뚜렷 |
| ★☆☆ | 관광형 자치구(종로·용산) 포지셔닝 | 외국인 관광객 타겟팅 고RevPAR 구조 |

**다음 단계:** `02_host_preprocessing.ipynb` → 피처 엔지니어링 & 모델 입력 준비